In [1]:
!pip install polars

import polars as pl

In [2]:
import os
import random
import sys
import cv2
import tqdm
import json
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import multilabel_confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [3]:

!nvidia-smi

Sun Apr 26 10:58:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               On  |   00000000:D1:00.0 Off |                  Off |
| 34%   62C    P2            150W /  230W |   15868MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os
os.chdir('/workspace')

In [5]:
import pandas as pd
import os

BASE_PATH = 'vindr_mammogram'
meta_path = os.path.join(BASE_PATH, 'metadata.csv')

device = pd.read_csv(meta_path)
device.head()

,SOP Instance UID,Series Instance UID,SOP Instance UID.1,Patient's Age,View Position,Image Laterality,Photometric Interpretation,Rows,Columns,Imager Pixel Spacing,...,Pixel Padding Value,Pixel Padding Range Limit,Window Center,Window Width,Rescale Intercept,Rescale Slope,Rescale Type,Window Center & Width Explanation,Manufacturer,Manufacturer's Model Name
0,d8125545210c08e1b1793a5af6458ee2,b36517b9cbbcfd286a7ae04f643af97a,d8125545210c08e1b1793a5af6458ee2,053Y,CC,L,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1662,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
1,290c658f4e75a3f83ec78a847414297c,b36517b9cbbcfd286a7ae04f643af97a,290c658f4e75a3f83ec78a847414297c,053Y,MLO,L,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1664,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
2,cd0fc7bc53ac632a11643ac4cc91002a,b36517b9cbbcfd286a7ae04f643af97a,cd0fc7bc53ac632a11643ac4cc91002a,053Y,CC,R,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1600,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
3,71638b1e853799f227492bfb08a01491,b36517b9cbbcfd286a7ae04f643af97a,71638b1e853799f227492bfb08a01491,053Y,MLO,R,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1654,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration
4,dd9ce3288c0773e006a294188aadba8e,d931832a0815df082c085b6e09d20aac,dd9ce3288c0773e006a294188aadba8e,042Y,CC,L,MONOCHROME2,3518,2800,"[0.085, 0.085]",...,0,NaN,1580,1500,0,1,US,linear LUT,SIEMENS,Mammomat Inspiration


In [6]:
BASE_PATH = 'vindr_mammogram'
IMG_DIR = os.path.join(BASE_PATH, 'mammo_processed_merged1') 
csv_files = [f for f in os.listdir(IMG_DIR) if f.endswith('.csv')]

# Print results
if csv_files:
    print(f"Found {len(csv_files)} CSV file(s) in {IMG_DIR}:")
    for csv_file in csv_files:
        print(f"{csv_file}")
else:
    print(f"No CSV files found in {IMG_DIR}")


Found 1 CSV file(s) in vindr_mammogram/mammo_processed_merged1:
mammo_processed_merged.csv


In [7]:
df = pl.read_csv("vindr_mammogram/mammo_processed_merged1/mammo_processed_merged.csv")
df = df.to_pandas()
df["merged_image_path"] = (
    df["merged_image_path"]
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
)
density_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3}

df['density'] = df['cc_breast_density'].str[-1].map(density_mapping)
df.head()



,study_id,laterality,merged_image_path,cc_image_id,mlo_image_id,split,cc_image_path,mlo_image_path,has_cc_bbox,has_mlo_bbox,...,mlo_crop_coords,cc_breast_birads,mlo_breast_birads,cc_breast_density,mlo_breast_density,cc_finding_categories,mlo_finding_categories,cc_finding_birads,mlo_finding_birads,density
0,0025a5dc99fd5c742026f0b2b030d3e9,L,vindr_mammogram/mammo_processed_merged1/0025a5...,451562831387e2822923204cf8f0873e,2ddfad7286c2b016931ceccd1e2c7bbc,test,vindr_original_data/mammo_processed_cropped/00...,vindr_original_data/mammo_processed_cropped/00...,False,False,...,"(0, 0, 992, 2443)",BI-RADS 1,BI-RADS 1,DENSITY C,DENSITY C,['No Finding'],['No Finding'],NaN,NaN,2
1,0025a5dc99fd5c742026f0b2b030d3e9,R,vindr_mammogram/mammo_processed_merged1/0025a5...,fcf12c2803ba8dc564bf1287c0c97d9a,47c8858666bcce92bcbd57974b5ce522,test,vindr_original_data/mammo_processed_cropped/00...,vindr_original_data/mammo_processed_cropped/00...,False,False,...,"(11, 0, 1064, 2560)",BI-RADS 1,BI-RADS 1,DENSITY C,DENSITY C,['No Finding'],['No Finding'],NaN,NaN,2
2,0028fb2c7f0b3a5cb9a80cb0e1cdbb91,L,vindr_mammogram/mammo_processed_merged1/0028fb...,3704f91985dcbc69f6ac2803523d1ecb,7fc1f1bb8bb1a7efaf7104e49c4d8b86,training,vindr_original_data/mammo_processed_cropped/00...,vindr_original_data/mammo_processed_cropped/00...,False,False,...,"(0, 0, 935, 2486)",BI-RADS 2,BI-RADS 2,DENSITY C,DENSITY C,['No Finding'],['No Finding'],NaN,NaN,2
3,0028fb2c7f0b3a5cb9a80cb0e1cdbb91,R,vindr_mammogram/mammo_processed_merged1/0028fb...,c4ce68631bf70949570ded31a3c69e60,16e58fc1d65fa7587247e6224ee96527,training,vindr_original_data/mammo_processed_cropped/00...,vindr_original_data/mammo_processed_cropped/00...,False,False,...,"(13, 0, 953, 2508)",BI-RADS 2,BI-RADS 2,DENSITY C,DENSITY C,['No Finding'],['No Finding'],NaN,NaN,2
4,0034765af074f93ed33d5e8399355caf,L,vindr_mammogram/mammo_processed_merged1/003476...,68f09c18925a66ef2840d4a62f237b31,b664cf1e7c968896144a3a2005cd3eb4,training,vindr_original_data/mammo_processed_cropped/00...,vindr_original_data/mammo_processed_cropped/00...,False,False,...,"(0, 0, 961, 2802)",BI-RADS 2,BI-RADS 2,DENSITY C,DENSITY C,['No Finding'],['No Finding'],NaN,NaN,2


In [8]:
import pandas as pd

device_info = device[['SOP Instance UID', "Manufacturer's Model Name"]].copy()
device_info.columns = ['image_id', 'device_model']

df = df.merge(device_info, left_on='cc_image_id', right_on='image_id', how='left')
df = df.drop('image_id', axis=1)

device_mapping = {
    'Mammomat Inspiration': 0,
    'Planmed Nuance': 1,
    'GIOTTO CLASS': 2,
    'GIOTTO IMAGE 3DL': 2
}

df['device_id'] = df['device_model'].map(device_mapping)

print(df['device_model'].value_counts())
print(df['device_id'].value_counts())
print(df.shape)

device_model
Mammomat Inspiration    7621
Planmed Nuance          1898
GIOTTO CLASS             314
GIOTTO IMAGE 3DL         166
Name: count, dtype: int64
device_id
0    7621
1    1898
2     480
Name: count, dtype: int64
(9999, 27)


In [9]:
def get_combined_finding_6class(cc_findings, mlo_findings, cc_birads, mlo_birads):
    if isinstance(cc_findings, str):
        cc_findings = eval(cc_findings) if cc_findings else []
    if isinstance(mlo_findings, str):
        mlo_findings = eval(mlo_findings) if mlo_findings else []
    
    cc_findings = cc_findings or []
    mlo_findings = mlo_findings or []
    all_findings = set(cc_findings + mlo_findings)
    
    if len(all_findings) > 1 and 'No Finding' in all_findings:
        all_findings.remove('No Finding')
    
    high_suspicion_structural = {
        'Architectural Distortion',
        'Skin Thickening',
        'Skin Retraction',
        'Nipple Retraction'
    }
    
    asymmetry_findings = {
        'Focal Asymmetry',
        'Global Asymmetry',
        'Asymmetry'
    }
    
    has_mass = 'Mass' in all_findings
    has_calc = 'Suspicious Calcification' in all_findings
    has_structural = bool(all_findings & high_suspicion_structural)
    has_asymmetry = bool(all_findings & asymmetry_findings)
    has_lymph = 'Suspicious Lymph Node' in all_findings
    
    def parse_birads(birads_str):
        if pd.isna(birads_str) or birads_str == '':
            return 0
        if isinstance(birads_str, str):
            try:
                return int(birads_str.strip().split()[-1])
            except:
                return 0
        return int(birads_str)
    
    cc_birads_num = parse_birads(cc_birads)
    mlo_birads_num = parse_birads(mlo_birads)
    max_birads = max(cc_birads_num, mlo_birads_num)
    
    if not all_findings or all_findings == {'No Finding'}:
        if max_birads == 1:
            return 0
        elif max_birads == 2:
            return 0
        else:
            return 1 if max_birads == 3 else 4
    
    if has_structural:
        return 4
    
    if has_mass and has_calc:
        return 3
    
    if has_mass:
        return 2
    
    if has_calc:
        return 1
    
    if has_lymph:
        return 3
    
    if has_asymmetry and len(all_findings) == 1:
        return -1
    
    if has_asymmetry and len(all_findings) > 1:
        return 4
    
    print(f"Warning: Unknown finding combination: {all_findings}, BIRADS: {max_birads}")
    return 4

df['finding'] = df.apply(
    lambda row: get_combined_finding_6class(
        row['cc_finding_categories'], 
        row['mlo_finding_categories'],
        row['cc_breast_birads'],
        row['mlo_breast_birads']
    ),
    axis=1
)
df.drop(df[df['finding'] == -1].index, inplace=True)
df['finding'].value_counts().sort_index()

finding
0    9032
1     132
2     490
3      66
4      90
Name: count, dtype: int64

In [10]:
import ast
import pandas as pd
from collections import Counter

ASYMMETRY_FINDINGS  = frozenset({"Asymmetry", "Focal Asymmetry", "Global Asymmetry"})
STRUCTURAL_FINDINGS = frozenset({"Architectural Distortion", "Skin Thickening", "Skin Retraction", "Nipple Retraction"})
FINDING_TO_BINARY   = [0, 1, 1, 1, 1, 1]
NUM_PROTOTYPES      = 6

def _parse_findings(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return frozenset()
    if isinstance(raw, (list, tuple, set)):
        return frozenset(str(f).strip() for f in raw if str(f).strip())
    if isinstance(raw, str):
        raw = raw.strip()
        if not raw:
            return frozenset()
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, (list, tuple, set)):
                return frozenset(str(f).strip() for f in parsed if str(f).strip())
        except (ValueError, SyntaxError):
            pass
        return frozenset({raw})
    return frozenset()

def _parse_birads(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return 0
    if isinstance(raw, (int, float)):
        return int(raw)
    s = str(raw).strip()
    for token in reversed(s.split()):
        digits = "".join(c for c in token if c.isdigit())
        if digits:
            return int(digits[0])
    return 0

def add_finding_columns(df,
                        cc_findings_col="cc_finding_categories",
                        mlo_findings_col="mlo_finding_categories",
                        cc_birads_col="cc_breast_birads",
                        mlo_birads_col="mlo_breast_birads"):
    rows, drop_idx, other_log = [], [], []

    for idx, row in df.iterrows():
        cc       = _parse_findings(row[cc_findings_col])
        mlo      = _parse_findings(row[mlo_findings_col])
        combined = cc | mlo

        if len(combined) > 1 and "No Finding" in combined:
            combined = combined - {"No Finding"}

        non_asym = combined - ASYMMETRY_FINDINGS
        if combined and not non_asym:
            drop_idx.append(idx)
            continue
        combined = non_asym

        max_birads  = max(_parse_birads(row[cc_birads_col]), _parse_birads(row[mlo_birads_col]))
        is_negative = not combined or combined == {"No Finding"}

        KNOWN = {"No Finding", "Mass", "Suspicious Calcification", "Suspicious Lymph Node"}
        remaining = combined - KNOWN - STRUCTURAL_FINDINGS

        if not is_negative and remaining:
            other_log.extend(list(remaining))

        rows.append({
            "idx":                idx,
            "has_neg_b1":         int(is_negative and max_birads <= 1),
            "has_neg_b2":         int(is_negative and max_birads > 1),
            "has_mass":           int("Mass" in combined),
            "has_calc":           int("Suspicious Calcification" in combined),
            "has_structural":     int(bool(combined & STRUCTURAL_FINDINGS)),
            "has_lymph": int(not is_negative and (
                                      "Suspicious Lymph Node" in combined or bool(remaining))),
        })

    print("=== Top findings landing in has_lymph_or_other ===")
    for finding, count in Counter(other_log).most_common(20):
        print(f"  {finding}: {count}")

    encoded = pd.DataFrame(rows).set_index("idx")
    df = df.drop(index=drop_idx).join(encoded).reset_index(drop=True)
    return df

df = add_finding_columns(df)

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph"]
print("\n=== Finding counts ===")
print(df[FINDING_COLS].sum())


=== Top findings landing in has_lymph_or_other ===

=== Finding counts ===
has_neg_b1        6703
has_neg_b2        2329
has_mass           570
has_calc           207
has_structural      90
has_lymph           22
dtype: int64


In [11]:
inbreast_df = pd.read_csv("inbreast_data/INbreast_merged/merged_metadata.csv")
inbreast_df["merged_image_path"] = (
    inbreast_df["merged_image_path"]
    .str.replace("INbreast Release 1.0", "inbreast_data", regex=False)
    .str.replace("vindr_original_data", "inbreast_data", regex=False))
inbreast_df['birads'] = inbreast_df['birads'].replace({'4a': '4', '4b': '4', '4c': '4','6':'5'})
inbreast_df['birads'] = (inbreast_df['birads'].astype(int) - 1).astype(int)
inbreast_df['density'] = 0
inbreast_df['finding'] = 0
inbreast_df['cc_breast_birads'] = None
inbreast_df['mlo_breast_birads'] = None
inbreast_df['cc_breast_density'] = None
inbreast_df['device_id'] = 0
for col in ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph", "has_other"]:
    inbreast_df[col] = 0
inbreast_df.head()

,patient_id,laterality,merged_image_path,cc_file_name,mlo_file_name,cc_image_path,mlo_image_path,birads,cc_roi_width,cc_roi_height,...,mlo_breast_birads,cc_breast_density,device_id,has_neg_b1,has_neg_b2,has_mass,has_calc,has_structural,has_lymph,has_other
0,024ee3569b2605dc,L,inbreast_data/INbreast_merged/024ee3569b2605dc...,20588020,20588072,INbreast Release 1.0/INbreast_processed/205880...,INbreast Release 1.0/INbreast_processed/205880...,1,1557,3231,...,None,None,0,0,0,0,0,0,0,0
1,024ee3569b2605dc,R,inbreast_data/INbreast_merged/024ee3569b2605dc...,20587994,20588046,INbreast Release 1.0/INbreast_processed/205879...,INbreast Release 1.0/INbreast_processed/205880...,4,1535,3128,...,None,None,0,0,0,0,0,0,0,0
2,069212ec65a94339,L,inbreast_data/INbreast_merged/069212ec65a94339...,50994787,50994733,INbreast Release 1.0/INbreast_processed/509947...,INbreast Release 1.0/INbreast_processed/509947...,0,1226,2580,...,None,None,0,0,0,0,0,0,0,0
3,069212ec65a94339,R,inbreast_data/INbreast_merged/069212ec65a94339...,50994706,50994760,INbreast Release 1.0/INbreast_processed/509947...,INbreast Release 1.0/INbreast_processed/509947...,0,1128,2566,...,None,None,0,0,0,0,0,0,0,0
4,0b7396cdccacca82,L,inbreast_data/INbreast_merged/0b7396cdccacca82...,22670832,22670878,INbreast Release 1.0/INbreast_processed/226708...,INbreast Release 1.0/INbreast_processed/226708...,1,1627,2983,...,None,None,0,0,0,0,0,0,0,0


In [12]:

def birads_to_binary(birads):
    if birads ==0:
        return 0 
    else:
        return 1 
inbreast_df['label'] = inbreast_df['birads'].apply(birads_to_binary)


In [13]:
def birads_to_label(birads_category):
    """Convert BI-RADS categories to numerical labels 0-4 (for 5 classes)"""
    birads_num = int(birads_category.replace(" ", "")[-1])
    return birads_num - 1
df['birads'] = df['cc_breast_birads'].apply(birads_to_label)

In [14]:
def birads_to_binary(birads):
    return 0 if birads in ['BI-RADS 1'] else 1 
df['label'] = df['cc_breast_birads'].apply(birads_to_binary)

In [15]:
inbreast_df['label'].value_counts()

label
1    157
0     30
Name: count, dtype: int64

In [16]:
df['cc_breast_density'].value_counts()

cc_breast_density
DENSITY C    7486
DENSITY D    1335
DENSITY B     942
DENSITY A      47
Name: count, dtype: int64

In [17]:
data = df[df['split'] == 'training']
test_df = df[df['split'] == 'test']

In [18]:
from sklearn.model_selection import train_test_split


study_meta = (
    data
    .groupby('study_id')
    .agg({
        'cc_breast_birads': 'first',   # BI-RADS at study level
        'finding': 'first'             # finding already encoded as 0–4
    })
    .reset_index()
)


# -------------------------------------------------
study_meta['stratify_key'] = (
    study_meta['cc_breast_birads'].astype(str) + '_' +
    study_meta['finding'].astype(str)
)


train_studies, val_studies = train_test_split(
    study_meta['study_id'],
    test_size=0.1,
    stratify=study_meta['stratify_key'],
    random_state=423
)

train_df = data[data['study_id'].isin(train_studies)].copy()
val_df   = data[data['study_id'].isin(val_studies)].copy()


In [19]:
train_df['density'].value_counts()

density
2    5396
3     963
1     668
0      30
Name: count, dtype: int64

In [20]:
val_df['density'].value_counts()

density
2    589
3    105
1     86
0      7
Name: count, dtype: int64

In [21]:
train_df.shape

(7057, 36)

In [22]:
import numpy as np
import cv2
from PIL import Image
import torchvision.transforms as transforms
import random
import torch


def get_transforms(img_size=(512, 512)):
    """Enhanced mammogram preprocessing with medical imaging considerations"""
    
    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        
        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=10,
                translate=(0.1, 0.1),
                scale=(0.9, 1.1),
                shear=6
            )
        ], p=0.6),
        
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([
            transforms.RandomPerspective(distortion_scale=0.1, p=1.0)
        ], p=0.1),
        
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=5.0, sigma=3.0)
        ], p=0.2),
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.95, 1.05),
                contrast=(0.9, 1.1)
            )
        ], p=0.4),
        transforms.Lambda(lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2)) 
                         if random.random() < 0.4 else x),
        

        
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
        ], p=0.2),
        
                # NOISE AUGMENTATIONS
        transforms.Lambda(lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.02)) 
                         if random.random() < 0.4 else x),
        
        transforms.Lambda(lambda x: add_speckle_noise(x, std=random.uniform(0.01, 0.03)) 
                         if random.random() < 0.2 else x),
        transforms.ToTensor(),
        
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    return train_transform, val_transform

def add_gaussian_noise(image, mean=0, std=0.02):
    """Gaussian noise - electronic noise in imaging sensors"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(mean, std, img_array.shape)
        noisy_img = np.clip(img_array + noise, 0, 1)
        return Image.fromarray((noisy_img * 255).astype(np.uint8))
    return image


def add_speckle_noise(image, std=0.03):
    """Speckle noise - multiplicative noise common in mammography"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(0, std, img_array.shape)
        noisy_img = img_array + img_array * noise
        return Image.fromarray((np.clip(noisy_img, 0, 1) * 255).astype(np.uint8))
    return image

def adjust_gamma(image, gamma=1.0):
    """
    Gamma correction - handles tissue density variation
    Gamma < 1 = brighter, > 1 = darker
    """
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        gamma_corrected = np.power(img_array, gamma)
        return Image.fromarray((gamma_corrected * 255).astype(np.uint8))
    return image

In [23]:
import cv2
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from PIL import Image
import os
from torchvision import transforms
import matplotlib.pyplot as plt
import random


class VinDrMammoDataset(Dataset):
    def __init__(self, dataframe, val_transform, mode="val"):
        self.df = dataframe.reset_index(drop=True)
        self.val_transform = val_transform
        self.mode = mode
        counts = self.df["label"].value_counts().to_dict()
        print(f"{mode.upper()} class distribution:", counts)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["merged_image_path"]).convert("RGB")
        label = int(row["label"])
        birads = row['birads']

        image_tensor = self.val_transform(image)

        density = row['density'] if 'density' in row else 0
        density_tensor = torch.tensor(density, dtype=torch.long)

        finding = row['finding'] if 'finding' in row else ""
        device_id = row['device_id'] if 'device_id' in row else ""

        return (
            row["merged_image_path"],
            image_tensor,
            torch.tensor(label, dtype=torch.long),
            torch.tensor(birads, dtype=torch.long) if isinstance(birads, (int, float)) else birads,
            density_tensor,
            finding,
            device_id
        )


def get_val_transform(img_size=(512, 512)):
    return transforms.Compose([
        transforms.Resize(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])


def create_test_loaders(val_df, test_df, inbreast_df, batch_size=8):
    """
    Create val, test, inbreast loaders at both 512 and 1024 resolutions.
    Returns:
        val_512, val_1024,
        test_512, test_1024,
        inbreast_512, inbreast_1024
    """

    transform_512  = get_val_transform((512, 512))
    transform_1024 = get_val_transform((1024, 1024))

    # ---- Datasets ----
    val_dataset_512      = VinDrMammoDataset(val_df,      transform_512,  mode="val")
    val_dataset_1024     = VinDrMammoDataset(val_df,      transform_1024, mode="val")

    test_dataset_512     = VinDrMammoDataset(test_df,     transform_512,  mode="val")
    test_dataset_1024    = VinDrMammoDataset(test_df,     transform_1024, mode="val")

    inbreast_dataset_512  = VinDrMammoDataset(inbreast_df, transform_512,  mode="val")
    inbreast_dataset_1024 = VinDrMammoDataset(inbreast_df, transform_1024, mode="val")

    # ---- DataLoaders ----
    val_kwargs      = dict(batch_size=batch_size, shuffle=False, num_workers=12, pin_memory=True)
    single_kwargs   = dict(batch_size=1,          shuffle=False, num_workers=12, pin_memory=True)

    val_512       = DataLoader(val_dataset_512,       **val_kwargs)
    val_1024      = DataLoader(val_dataset_1024,      **val_kwargs)

    test_512      = DataLoader(test_dataset_512,      **single_kwargs)
    test_1024     = DataLoader(test_dataset_1024,     **single_kwargs)

    inbreast_512  = DataLoader(inbreast_dataset_512,  **single_kwargs)
    inbreast_1024 = DataLoader(inbreast_dataset_1024, **single_kwargs)

    return val_512, val_1024, test_512, test_1024, inbreast_512, inbreast_1024



val_512, val_1024, test_512, test_1024, inbreast_512, inbreast_1024 = create_test_loaders(
    val_df, test_df, inbreast_df, batch_size=8
)


for batch in test_512:
    path, image, label, birads, density, finding, device_id = batch
    print("Path      :", path[0])
    print("Image     :", image.shape)       
    print("Label     :", label)
    print("BiRADS    :", birads)
    print("Density   :", density)
    print("Finding   :", finding[0])
    print("Device ID :", device_id[0])
    break

VAL class distribution: {0: 539, 1: 248}
VAL class distribution: {0: 539, 1: 248}
VAL class distribution: {0: 1341, 1: 625}
VAL class distribution: {0: 1341, 1: 625}
VAL class distribution: {1: 157, 0: 30}
VAL class distribution: {1: 157, 0: 30}
Path      : vindr_mammogram/mammo_processed_merged1/0025a5dc99fd5c742026f0b2b030d3e9/0025a5dc99fd5c742026f0b2b030d3e9_L_merged.png
Image     : torch.Size([1, 3, 512, 512])
Label     : tensor([0])
BiRADS    : tensor([0])
Density   : tensor([2])
Finding   : tensor(0)
Device ID : tensor(0)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import os
import gc
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    confusion_matrix, precision_score, recall_score,
    roc_curve, auc as sklearn_auc, classification_report
)
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

NUM_CLASSES = 5
NUM_FINDINGS = 6
IMG_SIZE = 512
BATCH_SIZE = 8

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph"]

FINDING_MAP = {
    0: 'Negative BI-RADS 1',
    1: 'Negative BI-RADS 2',
    2: 'Mass',
    3: 'Calcification',
    4: 'Structural/Complex',
    5: 'Lymphadenopathy',
}

DEVICE_MAP = {
    0: 'Mammomat Inspiration',
    1: 'Planmed Nuance',
    2: 'GIOTTO',
}

DENSITY_MAP = {
    0: 'Density A (Fatty)',
    1: 'Density B (Scattered)',
    2: 'Density C (Heterogeneous)',
    3: 'Density D (Dense)',
}

MODEL_CHECKPOINTS = {
    'efficientnet_b3': '/workspace/Thesis_updated_results/birads_512_dual_proto/efficientnet_b3/best_model.pth',
    'convnext_base': '/workspace/Thesis_updated_results/birads_512_dual_proto/convnext_base/best_model.pth',
}

OUTPUT_BASE_DIR = '/workspace/Thesis_updated_results/birads_analysis_results'


class MammoDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['merged_image_path']
        
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        
        image = self.transform(image)
        
        birads = int(row['label'])
        density = int(row['density'])
        device_id = int(row['device_id'])
        
        finding_vec = np.array([row[c] for c in FINDING_COLS], dtype=np.float32)
        
        return img_path, image, birads, density, device_id, finding_vec


class AttentionPool2d(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.attn = nn.Conv2d(c, 1, 1)

    def forward(self, x):
        w = F.softmax(self.attn(x).flatten(2), dim=-1)
        return (x.flatten(2) * w).sum(-1)


class Model(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        
        if isinstance(backbone, models.EfficientNet):
            self.feat_dim = backbone.classifier[1].in_features
            backbone.classifier[1] = nn.Identity()
        else:
            self.feat_dim = backbone.classifier[2].in_features
            backbone.classifier[2] = nn.Identity()
        
        self.pool = AttentionPool2d(self.feat_dim)
        self.fc = nn.Linear(self.feat_dim, NUM_CLASSES - 1)

    def forward(self, x):
        f = self.backbone(x)
        if f.ndim == 4:
            f = self.pool(f)
        return self.fc(f)


def ordinal_to_label(logits):
    return (torch.sigmoid(logits) > 0.5).sum(1)


def ordinal_to_prob(logits):
    q = torch.sigmoid(logits)
    B = q.size(0)
    K = q.size(1) + 1
    probs = torch.zeros(B, K).to(logits.device)
    
    probs[:,0] = 1 - q[:,0]
    for i in range(1, K-1):
        probs[:,i] = q[:,i-1] - q[:,i]
    probs[:,-1] = q[:,-1]
    
    return probs


@torch.no_grad()
def run_inference(model, loader):
    model.eval()
    
    rows = []
    
    for paths, imgs, y, d, dev, fvec in loader:
        imgs = imgs.to(DEVICE)
        
        logits = model(imgs)
        probs = ordinal_to_prob(logits)
        preds = ordinal_to_label(logits)
        
        for i in range(len(paths)):
            row = {
                'path': paths[i],
                'label': int(y[i]),
                'pred': int(preds[i]),
                'density': int(d[i]),
                'device_id': int(dev[i]),
            }
            
            for c in range(NUM_CLASSES):
                row[f'prob_cls{c}'] = probs[i,c].item()
            
            for j in range(NUM_FINDINGS):
                row[f'finding_{j}'] = fvec[i][j]
            
            rows.append(row)
    
    return pd.DataFrame(rows)


def compute_metrics(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Macro-F1': f1_score(y_true, y_pred, average='macro')
    }


def analyze_finding(df_res, output_dir):
    results = []
    
    for f_id in range(NUM_FINDINGS):
        col = f'finding_{f_id}'
        mask = df_res[col] > 0.5
        
        if mask.sum() < 1:
            continue
        
        subset = df_res[mask]
        y_true = subset['label'].values
        y_pred = subset['pred'].values
        
        total = len(subset)
        correct = (y_true == y_pred).sum()
        incorrect = (y_true != y_pred).sum()
        
        row = {
            'Finding': FINDING_MAP[f_id],
            'N': int(total),
            'Correct': int(correct),
            'Incorrect': int(incorrect),
        }
        
        if f_id >= 2:
            correct_mask = (y_true == y_pred)
            incorrect_mask = (y_true != y_pred)
            
            correct_high = np.sum(correct_mask & (y_pred >= 3))
            incorrect_low = np.sum(incorrect_mask & (y_pred <= 2))
            
            row.update({
                'Correct_BIRADS_3_4_5': int(correct_high),
                'Incorrect_to_BIRADS_1_2': int(incorrect_low),
            })
        
        row.update(compute_metrics(y_true, y_pred))
        results.append(row)
    
    df_out = pd.DataFrame(results)
    
    os.makedirs(output_dir, exist_ok=True)
    df_out.to_csv(os.path.join(output_dir, 'finding_analysis.csv'), index=False)
    
    print(df_out.to_string(index=False))
    return df_out


def run_analysis(df):
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor()
    ])
    
    loader = DataLoader(MammoDataset(df, transform), batch_size=BATCH_SIZE)
    
    for name, path in MODEL_CHECKPOINTS.items():
        if 'efficientnet' in name:
            backbone = models.efficientnet_b3(weights=None)
        else:
            backbone = models.convnext_base(weights=None)
        
        model = Model(backbone).to(DEVICE)
        ckpt = torch.load(path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'], strict=False)
        
        df_res = run_inference(model, loader)
        
        out_dir = os.path.join(OUTPUT_BASE_DIR, name)
        analyze_finding(df_res, out_dir)


if __name__ == "__main__":
    run_analysis(test_df)

           Finding    N  Correct  Incorrect  Accuracy  Macro-F1  Correct_BIRADS_3_4_5  Incorrect_to_BIRADS_1_2
Negative BI-RADS 1 1341     1341          0       1.0       1.0                   NaN                      NaN
Negative BI-RADS 2  467        0        467       0.0       0.0                   NaN                      NaN
              Mass  113        0        113       0.0       0.0                   0.0                    113.0
     Calcification   50        0         50       0.0       0.0                   0.0                     50.0
Structural/Complex   15        0         15       0.0       0.0                   0.0                     15.0
   Lymphadenopathy    3        0          3       0.0       0.0                   0.0                      3.0


In [ ]:
sssssssss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import os
import glob

# ─────────────────────────────────────────────────────────────────────────────
# 1. Generic Model Network
# ─────────────────────────────────────────────────────────────────────────────

class GenericNet(nn.Module):
    """Generic network for binary classification"""
    
    def __init__(self, backbone_name, num_classes=2):
        super().__init__()
        self.backbone_name = backbone_name
        self.num_classes = num_classes
        
        if 'efficientnet_b3' in backbone_name:
            self.backbone = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
            num_features = self.backbone.classifier[1].in_features
            self.backbone.classifier = nn.Identity()
            
        elif 'convnext_base' in backbone_name:
            self.backbone = models.convnext_base(weights=models.ConvNeXt_Base_Weights.DEFAULT)
            num_features = self.backbone.classifier[2].in_features
            self.backbone.classifier = nn.Identity()
        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")
        
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        if isinstance(features, tuple):
            features = features[0]
        if features.ndim == 4:
            features = self.global_pool(features).flatten(1)
        logits = self.classifier(features)
        return logits


# ─────────────────────────────────────────────────────────────────────────────
# 2. Inference Function
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def run_inference(model, dataloader, device):
    """Run inference and return predictions"""
    model.eval()
    all_labels = []
    all_preds = []
    all_probs = []
    
    for batch in dataloader:
        _, images, labels, _, _, _, _ = batch
        images = images.to(device)
        
        logits = model(images)
        probs = F.softmax(logits, dim=1)[:, 1]
        preds = logits.argmax(dim=1)
        
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


# ─────────────────────────────────────────────────────────────────────────────
# 3. Calculate Metrics
# ─────────────────────────────────────────────────────────────────────────────

def get_metrics(labels, preds, probs):
    """Calculate Precision, Recall, Macro-F1, AUC"""
    
    precision = precision_score(labels, preds, pos_label=1, zero_division=0)
    recall = recall_score(labels, preds, pos_label=1, zero_division=0)
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    
    try:
        auc = roc_auc_score(labels, probs)
    except:
        auc = np.nan
    
    return {
        'Precision': round(float(precision), 4),
        'Recall': round(float(recall), 4),
        'Macro-F1': round(float(macro_f1), 4),
        'AUC': round(float(auc), 4) if not np.isnan(auc) else '—'
    }


# ─────────────────────────────────────────────────────────────────────────────
# 4. Find Checkpoint Paths
# ─────────────────────────────────────────────────────────────────────────────

def find_checkpoints(base_path='Thesis_updated_results'):
    """Find checkpoint paths from directory structure"""
    
    models_config = []
    
    # Look for efficientnet_b3
    effnet_512 = glob.glob(f'{base_path}/**/efficientnet_b3/**/best_model.pth', recursive=True)
    effnet_1024 = glob.glob(f'{base_path}/**/efficientnet_b3/**/best_model.pth', recursive=True)
    
    # Look for convnext_base
    convnext_512 = glob.glob(f'{base_path}/**/convnext_base/**/best_model.pth', recursive=True)
    convnext_1024 = glob.glob(f'{base_path}/**/convnext_base/**/best_model.pth', recursive=True)
    
    # Check common paths
    effnet_512_path = None
    effnet_1024_path = None
    convnext_512_path = None
    convnext_1024_path = None
    
    # Try common directory structures
    common_paths = {
        'efficientnet_b3_512': [
            f'{base_path}/Merged_binary_512/efficientnet_b3/best_model.pth',
            f'{base_path}/binary_512/efficientnet_b3/best_model.pth',
        ],
        'efficientnet_b3_1024': [
            f'{base_path}/Merged_binary_1024/efficientnet_b3/best_model.pth',
            f'{base_path}/binary_1024/efficientnet_b3/best_model.pth',
        ],
        'convnext_base_512': [
            f'{base_path}/Merged_binary_512/convnext_base/best_model.pth',
            f'{base_path}/binary_512/convnext_base/best_model.pth',
        ],
        'convnext_base_1024': [
            f'{base_path}/Merged_binary_1024/convnext_base/best_model.pth',
            f'{base_path}/binary_1024/convnext_base/best_model.pth',
        ]
    }
    
    for key, paths in common_paths.items():
        for path in paths:
            if os.path.exists(path):
                print(f"✓ Found: {key} -> {path}")
                if '512' in key and 'efficientnet' in key:
                    effnet_512_path = path
                elif '1024' in key and 'efficientnet' in key:
                    effnet_1024_path = path
                elif '512' in key and 'convnext' in key:
                    convnext_512_path = path
                elif '1024' in key and 'convnext' in key:
                    convnext_1024_path = path
                break
    
    models_config = [
        {
            'name': 'efficientnet_b3',
            'ckpt_512': effnet_512_path,
            'ckpt_1024': effnet_1024_path
        },
        {
            'name': 'convnext_base',
            'ckpt_512': convnext_512_path,
            'ckpt_1024': convnext_1024_path
        }
    ]
    
    return models_config


# ─────────────────────────────────────────────────────────────────────────────
# 5. Main Evaluation
# ─────────────────────────────────────────────────────────────────────────────

def evaluate_all_models(models_config, loaders_config, device='cuda'):
    """Evaluate all models and create summary table"""
    
    device = torch.device(device if torch.cuda.is_available() else 'cpu')
    print(f"\nDevice: {device}\n")
    
    all_results = []
    
    for model_cfg in models_config:
        model_name = model_cfg['name']
        print(f"\n{'='*70}")
        print(f"MODEL: {model_name}")
        print(f"{'='*70}")
        
        for resolution in ['512', '1024']:
            ckpt_key = f'ckpt_{resolution}'
            ckpt_path = model_cfg.get(ckpt_key)
            
            if not ckpt_path or not os.path.exists(ckpt_path):
                print(f"  [{resolution}] Checkpoint not found: {ckpt_path}")
                continue
            
            # Load model
            print(f"  [{resolution}] Loading checkpoint...")
            model = GenericNet(model_name).to(device)
            try:
                checkpoint = torch.load(ckpt_path, map_location=device)
                model.load_state_dict(checkpoint)
                print(f"  [{resolution}] ✓ Loaded successfully")
            except Exception as e:
                print(f"  [{resolution}] ✗ Error: {e}")
                continue
            
            # Evaluate on each split
            for split_name in ['val', 'test', 'inbreast']:
                loader_key = f'{split_name}_{resolution}'
                loader = loaders_config.get(loader_key)
                
                if loader is None:
                    print(f"    [{split_name}] Loader not found: {loader_key}")
                    continue
                
                print(f"    Evaluating {split_name} ({resolution}px)...", end=' ')
                
                try:
                    labels, preds, probs = run_inference(model, loader, device)
                    metrics = get_metrics(labels, preds, probs)
                    
                    result = {
                        'Model': model_name,
                        'Split': split_name.upper(),
                        'Resolution': resolution,
                        'N': len(labels),
                        **metrics
                    }
                    all_results.append(result)
                    print(f"✓ AUC={metrics['AUC']}")
                except Exception as e:
                    print(f"✗ Error: {e}")
                    continue
            
            del model
            torch.cuda.empty_cache()
    
    # Create summary DataFrame
    results_df = pd.DataFrame(all_results)
    
    print(f"\n{'='*70}")
    print("SUMMARY TABLE - MODEL RESULTS")
    print(f"{'='*70}\n")
    print(results_df.to_string(index=False))
    
    # Save to CSV
    os.makedirs('results', exist_ok=True)
    csv_path = 'results/model_results_summary.csv'
    results_df.to_csv(csv_path, index=False)
    print(f"\n✓ Saved to: {csv_path}")
    
    return results_df


# ─────────────────────────────────────────────────────────────────────────────
# 6. RUN EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':
    
    # Find checkpoints automatically
    print("Finding checkpoints...")
    models_config = find_checkpoints(base_path='Thesis_updated_results')
    
    # Prepare loaders
    loaders_config = {
        'val_512': val_512,
        'val_1024': val_1024,
        'test_512': test_512,
        'test_1024': test_1024,
        'inbreast_512': inbreast_512,
        'inbreast_1024': inbreast_1024
    }
    
    # Run evaluation
    results_df = evaluate_all_models(models_config, loaders_config, device='cuda')
    
    print("\n" + "="*70)
    print("EVALUATION COMPLETE!")
    print("="*70)